# Stochastic Event Set from IDF/RP Maps
## Preserving spatial dependence via empirical variogram + Gaussian copula

**Theory:**  
IDF/RP maps give marginal distributions at each cell.  
To generate physically plausible *spatial* events we need the **joint distribution**  
across cells — i.e. the spatial copula.

**Workflow:**
1. Load annual-maximum stacks (n_years × lat × lon) from the IDF pipeline  
2. Transform each cell to uniform margins via the empirical CDF (rank transform)  
3. Estimate the **spatial variogram** of the Gaussian-transformed fields  
4. Fit a parametric variogram model (exponential or Matérn)  
5. Sample from the fitted **Gaussian copula** using the Cholesky covariance matrix  
6. Back-transform each sample through the **pixel-wise GEV inverse CDF**  
7. Each sample is one spatially coherent "what could happen" event field

**Spatial validity limits:**
| Condition | Limit |
|---|---|
| Variogram stationarity (flat terrain) | 50–200 km |
| Variogram stationarity (orographic) | 20–80 km |
| Cholesky factorisation (memory) | ~500 cells → use sub-domain |
| Physical realism of simulated events | Within one climate zone |

**References:**
- Bárdossy & Pegram (2009) – spatial downscaling with copulas  
- Serinaldi (2013) – stochastic rainfall using copulas  
- Gräler et al. (2016) – gstat variogram fitting  


## 1. Imports & configuration

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from scipy.stats import genextreme, rankdata, norm
from scipy.optimize import curve_fit
from tqdm.auto import tqdm

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    print('cartopy not found – maps will use pcolormesh')

print('All imports OK')


## 2. User configuration

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
IDF_DIR    = '/home/admin_climatecharted_com/data/MOloch/IDF_results'
OUTPUT_DIR = os.path.join(IDF_DIR, 'stochastic_events')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── DURATION & TARGET DOMAIN ──────────────────────────────────────────────────
DURATION_H  = 3        # hours  (must match an annmax_<d>h.nc file)
YEARS       = list(range(1991, 2021))   # 30 years of annual maxima

# Target sub-domain for copula simulation
# Keep this ≤ ~100 km × 100 km for stationarity and memory reasons
DOMAIN_LAT_MIN = 44.5
DOMAIN_LAT_MAX = 46.5
DOMAIN_LON_MIN = 8.0
DOMAIN_LON_MAX = 11.0

# Sub-sampling: use every N-th grid cell to keep Cholesky feasible
# For 960 × 768 at ~2.5 km spacing: N=8 → ~480 m effective spacing, ~10 000 cells
# For domain ~100 × 100 km at 2.5 km: ~1 600 cells at N=1, safe for Cholesky
SUBSAMPLE_N = 4   # increase if memory is tight

# ── VARIOGRAM ─────────────────────────────────────────────────────────────────
# Maximum lag distance for variogram estimation [km]
VARIOGRAM_MAX_LAG_KM  = 150.0
# Number of lag bins
VARIOGRAM_N_BINS      = 30
# Variogram model: 'exponential' or 'matern15' (Matérn ν=1.5)
VARIOGRAM_MODEL       = 'exponential'

# ── SIMULATION ────────────────────────────────────────────────────────────────
N_SIMULATIONS         = 1000   # number of stochastic events to generate
TARGET_RP             = 100    # return period to inspect in validation plots
RANDOM_SEED           = 42

print(f'Duration       : {DURATION_H}h')
print(f'Domain         : lat [{DOMAIN_LAT_MIN}, {DOMAIN_LAT_MAX}]  '
      f'lon [{DOMAIN_LON_MIN}, {DOMAIN_LON_MAX}]')
print(f'N simulations  : {N_SIMULATIONS}')
print(f'Subsample N    : {SUBSAMPLE_N}')


## 3. Load annual-maximum stack

In [ ]:
annmax_path = os.path.join(IDF_DIR, f'annmax_{DURATION_H}h.nc')
if not os.path.exists(annmax_path):
    raise FileNotFoundError(
        f'{annmax_path} not found.\n'
        f'Run MORE_precipitation_IDF_Italy.ipynb to generate annual-max files.'
    )

ds_am  = xr.open_dataset(annmax_path)
var_am = f'annmax_tp_{DURATION_H}h'
if var_am not in ds_am:
    var_am = list(ds_am.data_vars)[0]

da_am = ds_am[var_am]   # shape: (year, lat, lon) or (n_years, lat, lon)
print(f'Annual-max variable : {var_am}')
print(f'Shape               : {da_am.shape}')
print(f'Dimensions          : {da_am.dims}')

lat_full = da_am['lat'].values
lon_full = da_am['lon'].values
print(f'Lat range : {lat_full.min():.3f} – {lat_full.max():.3f}')
print(f'Lon range : {lon_full.min():.3f} – {lon_full.max():.3f}')


## 4. Clip to target sub-domain

In [ ]:
# ── Clip spatially ────────────────────────────────────────────────────────────
lat_mask = (lat_full >= DOMAIN_LAT_MIN) & (lat_full <= DOMAIN_LAT_MAX)
lon_mask = (lon_full >= DOMAIN_LON_MIN) & (lon_full <= DOMAIN_LON_MAX)

da_domain = da_am.isel(lat=np.where(lat_mask)[0], lon=np.where(lon_mask)[0]).load()
ds_am.close()

lat_d = da_domain['lat'].values
lon_d = da_domain['lon'].values
print(f'Domain shape (years, lat, lon) : {da_domain.shape}')

# ── Sub-sample for Cholesky feasibility ───────────────────────────────────────
da_sub = da_domain.isel(
    lat=slice(None, None, SUBSAMPLE_N),
    lon=slice(None, None, SUBSAMPLE_N),
)
lat_s = da_sub['lat'].values
lon_s = da_sub['lon'].values
n_lat_s, n_lon_s = len(lat_s), len(lon_s)
n_years = da_sub.shape[0]
n_cells = n_lat_s * n_lon_s

print(f'Sub-sampled shape : {da_sub.shape}')
print(f'Number of cells   : {n_cells}  (Cholesky = {n_cells}² = {n_cells**2:,} elements)')

# Domain diagonal in km
KM_PER_LAT = 111.0
KM_PER_LON = 111.0 * np.cos(np.radians(np.mean(lat_d)))
diag_km = np.sqrt(
    ((lat_d.max() - lat_d.min()) * KM_PER_LAT)**2 +
    ((lon_d.max() - lon_d.min()) * KM_PER_LON)**2
)
print(f'Domain diagonal   : {diag_km:.0f} km')


## 5. Transform annual maxima to uniform margins (rank transform)

The copula is defined on **uniform marginals U(0,1)**.  
We use the empirical rank transform (Hazen plotting position) to be non-parametric.  
The Gaussian copula then works in the Gaussian-transformed space.


In [ ]:
# annmax_stack shape: (n_years, n_lat_s, n_lon_s)
annmax_stack = da_sub.values.astype(np.float32)

n_y, n_ls, n_lo = annmax_stack.shape
n_cells = n_ls * n_lo

# Reshape to (n_years, n_cells) for vectorised operations
X = annmax_stack.reshape(n_y, n_cells)   # (30, n_cells)

# ── Rank transform → uniform [0,1] margins ────────────────────────────────────
# Hazen: u = (rank - 0.5) / n
U = np.zeros_like(X)
for j in range(n_cells):
    col = X[:, j]
    valid = ~np.isnan(col)
    if valid.sum() >= 3:
        r = rankdata(col[valid])
        u = (r - 0.5) / valid.sum()
        U[valid, j] = u
    else:
        U[:, j] = np.nan

# ── Normal-score transform → Gaussian margins ─────────────────────────────────
# Clip slightly away from 0/1 to avoid infinite ppf
U_clipped = np.clip(U, 1e-6, 1 - 1e-6)
Z = norm.ppf(U_clipped)   # Gaussian-transformed field, (n_years, n_cells)

# NaN propagation
Z[np.isnan(X)] = np.nan

print(f'Uniform margins   U : min={np.nanmin(U):.3f}  max={np.nanmax(U):.3f}')
print(f'Gaussian margins  Z : min={np.nanmin(Z):.2f}  max={np.nanmax(Z):.2f}')
print(f'Expected ~N(0,1)  →  mean={np.nanmean(Z):.3f}  std={np.nanstd(Z):.3f}')


## 6. Empirical variogram estimation

The **variogram** γ(h) = ½ E[(Z(x) − Z(x+h))²] describes how spatial correlation  
decays with lag distance h.  We estimate it from the n_years realizations of the  
normal-score field.


In [ ]:
# ── Build coordinate arrays for all cells ─────────────────────────────────────
LON_G, LAT_G = np.meshgrid(lon_s, lat_s)
x_km = LON_G.ravel() * KM_PER_LON   # project to km (approximate, local)
y_km = LAT_G.ravel() * KM_PER_LAT

# ── Compute empirical variogram ────────────────────────────────────────────────
# For each pair of years: compute all pairwise γ values and bin by distance
# Use a random subsample of cell pairs if n_cells is large

rng = np.random.default_rng(RANDOM_SEED)

# Sample up to 5000 cell pairs for speed
MAX_PAIRS = 5000
if n_cells * (n_cells - 1) // 2 > MAX_PAIRS:
    idx_pairs = rng.choice(n_cells, size=(MAX_PAIRS * 2,), replace=True).reshape(-1, 2)
    idx_pairs = idx_pairs[idx_pairs[:, 0] != idx_pairs[:, 1]][:MAX_PAIRS]
else:
    i_idx, j_idx = np.triu_indices(n_cells, k=1)
    idx_pairs    = np.column_stack([i_idx, j_idx])

i_idx = idx_pairs[:, 0]
j_idx = idx_pairs[:, 1]

# Lag distances
dists = np.sqrt((x_km[i_idx] - x_km[j_idx])**2 +
                (y_km[i_idx] - y_km[j_idx])**2)

# γ(h) = 0.5 * mean over years of (Z_i - Z_j)^2
diff2 = 0.5 * np.nanmean((Z[:, i_idx] - Z[:, j_idx])**2, axis=0)

# ── Bin into lag classes ───────────────────────────────────────────────────────
bins       = np.linspace(0, VARIOGRAM_MAX_LAG_KM, VARIOGRAM_N_BINS + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])
gamma_emp   = np.full(VARIOGRAM_N_BINS, np.nan)
n_pairs_bin = np.zeros(VARIOGRAM_N_BINS, dtype=int)

for k in range(VARIOGRAM_N_BINS):
    mask = (dists >= bins[k]) & (dists < bins[k + 1])
    if mask.sum() >= 3:
        gamma_emp[k] = np.nanmean(diff2[mask])
        n_pairs_bin[k] = mask.sum()

valid_bins = ~np.isnan(gamma_emp)
print(f'Variogram bins populated: {valid_bins.sum()} / {VARIOGRAM_N_BINS}')
print(f'Sill estimate (max γ)   : {np.nanmax(gamma_emp):.3f}  (expected ~1.0 for N(0,1))')


## 7. Fit parametric variogram model

In [ ]:
def variogram_exponential(h, nugget, sill, range_km):
    """Exponential variogram: γ(h) = nugget + (sill-nugget)*(1 - exp(-h/range))"""
    return nugget + (sill - nugget) * (1.0 - np.exp(-h / range_km))

def variogram_matern15(h, nugget, sill, range_km):
    """Matérn ν=1.5: γ(h) = (sill-nugget)*(1 - (1 + h*√3/range)*exp(-h*√3/range)) + nugget"""
    u = np.sqrt(3) * h / range_km
    return nugget + (sill - nugget) * (1.0 - (1.0 + u) * np.exp(-u))

model_fn = variogram_exponential if VARIOGRAM_MODEL == 'exponential' else variogram_matern15

h_fit    = bin_centers[valid_bins]
gam_fit  = gamma_emp[valid_bins]

# Initial guesses
p0 = [0.05, np.nanmax(gam_fit) * 0.95, 30.0]
bounds = ([0, 0.1, 1.0], [0.5, 2.0, VARIOGRAM_MAX_LAG_KM])

try:
    popt, _ = curve_fit(model_fn, h_fit, gam_fit, p0=p0, bounds=bounds,
                        maxfev=5000)
    nugget_fit, sill_fit, range_fit = popt
    print(f'Variogram fit ({VARIOGRAM_MODEL}):')
    print(f'  Nugget : {nugget_fit:.4f}')
    print(f'  Sill   : {sill_fit:.4f}  (expected ~1.0)')
    print(f'  Range  : {range_fit:.1f} km  ← spatial correlation scale')
    print()
    # Practical range: distance where γ ≈ 0.95 × sill
    if VARIOGRAM_MODEL == 'exponential':
        practical_range = -np.log(0.05) * range_fit
    else:
        # Matérn 1.5: solve numerically
        from scipy.optimize import brentq
        f_pr = lambda h: model_fn(h, nugget_fit, sill_fit, range_fit) - 0.95 * sill_fit
        practical_range = brentq(f_pr, 1, VARIOGRAM_MAX_LAG_KM * 2)
    print(f'  Practical range (95% sill): {practical_range:.1f} km')
    print()
    print(f'VALIDITY LIMIT for stochastic simulation: {practical_range:.0f} km')
except Exception as e:
    print(f'Variogram fitting failed: {e}')
    print('Using default parameters — refine manually.')
    nugget_fit, sill_fit, range_fit = 0.05, 1.0, 40.0
    practical_range = 120.0

# ── Plot variogram ─────────────────────────────────────────────────────────────
h_plot = np.linspace(0, VARIOGRAM_MAX_LAG_KM, 300)
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(h_fit, gam_fit, s=n_pairs_bin[valid_bins] / n_pairs_bin[valid_bins].max() * 80 + 10,
           c='steelblue', alpha=0.7, label='Empirical γ(h)', zorder=3)
ax.plot(h_plot, model_fn(h_plot, nugget_fit, sill_fit, range_fit),
        'r-', lw=2, label=f'{VARIOGRAM_MODEL.capitalize()} fit
range={range_fit:.0f} km')
ax.axvline(practical_range, color='green', ls='--', lw=1.5,
           label=f'Practical range: {practical_range:.0f} km')
ax.axhline(sill_fit, color='gray', ls=':', lw=1.2, label=f'Sill={sill_fit:.3f}')
ax.set_xlabel('Lag distance [km]', fontsize=12)
ax.set_ylabel('γ(h)  [–]', fontsize=12)
ax.set_title(f'Empirical variogram — normal-score annual maxima — {DURATION_H}h', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'variogram_{DURATION_H}h.png'), dpi=150)
plt.show()
print('Variogram plot saved.')


## 8. Build spatial covariance matrix and Cholesky decomposition

The **covariance** C(h) = sill − γ(h) for a stationary intrinsically isotropic model.  
The Cholesky factor L satisfies C = L Lᵀ; samples are drawn as Z = L ξ where ξ ~ N(0,I).


In [ ]:
# ── Pairwise distances between all sub-sampled cells ─────────────────────────
# (n_cells × n_cells) distance matrix
dx = (x_km[:, None] - x_km[None, :])
dy = (y_km[:, None] - y_km[None, :])
H  = np.sqrt(dx**2 + dy**2)   # (n_cells, n_cells)

# ── Covariance matrix C = sill − γ(H) ────────────────────────────────────────
C = sill_fit - model_fn(H, nugget_fit, sill_fit, range_fit)
np.fill_diagonal(C, sill_fit)   # C(0) = sill (variance)

# ── Regularize and Cholesky-decompose ─────────────────────────────────────────
# Add small nugget to diagonal for numerical stability
jitter = max(nugget_fit, 1e-6) * np.eye(n_cells)
C_reg  = C + jitter

try:
    L = np.linalg.cholesky(C_reg)
    print(f'Cholesky OK — matrix size: {n_cells} × {n_cells}')
    print(f'Memory for C_reg: {C_reg.nbytes / 1e6:.1f} MB')
except np.linalg.LinAlgError as e:
    print(f'Cholesky failed ({e}); applying additional regularization')
    eigenvalues = np.linalg.eigvalsh(C_reg)
    min_eig = eigenvalues.min()
    C_reg  += (-min_eig + 1e-6) * np.eye(n_cells)
    L = np.linalg.cholesky(C_reg)
    print('Cholesky succeeded after regularization.')


## 9. Fit GEV parameters at each cell (for back-transformation)

In [ ]:
# ── GEV fitting on the sub-sampled grid ───────────────────────────────────────
GEV_XI_CAP = 0.5   # matches the IDF notebook convention

gev_params = np.full((n_cells, 3), np.nan)   # (shape_c, loc, scale) per cell

print('Fitting GEV at each cell ...')
for j in tqdm(range(n_cells)):
    ts = X[:, j]
    valid = ts[~np.isnan(ts)]
    if len(valid) < 5 or valid.max() == 0:
        continue
    try:
        c, loc, scale = genextreme.fit(valid)
        c = max(c, -GEV_XI_CAP)   # cap shape
        gev_params[j] = [c, loc, scale]
    except Exception:
        pass

n_fitted = np.sum(~np.isnan(gev_params[:, 0]))
print(f'GEV fitted successfully at {n_fitted} / {n_cells} cells ({100*n_fitted/n_cells:.0f}%)')


## 10. Generate stochastic event set

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

# ── Pre-allocate output (n_sims, n_cells) ─────────────────────────────────────
events = np.full((N_SIMULATIONS, n_cells), np.nan, dtype=np.float32)

print(f'Simulating {N_SIMULATIONS} events ...')
for s in tqdm(range(N_SIMULATIONS)):
    # 1. Sample standard normal field
    xi = rng.standard_normal(n_cells)
    # 2. Impose spatial covariance via Cholesky
    z_sim = L @ xi   # (n_cells,)  — correlated Gaussian field
    # 3. Transform to uniform via normal CDF
    u_sim = norm.cdf(z_sim)
    # 4. Back-transform through pixel-wise GEV quantile function
    for j in range(n_cells):
        if np.isnan(gev_params[j, 0]):
            continue
        c, loc, scale = gev_params[j]
        events[s, j] = float(genextreme.ppf(u_sim[j], c, loc=loc, scale=scale))

# Clip negative values (GEV can produce negatives for large shape)
events = np.clip(events, 0, None)

print(f'Events shape: {events.shape}')
print(f'Mean of all events : {np.nanmean(events):.1f} mm')
print(f'99th percentile    : {np.nanpercentile(events, 99):.1f} mm')
print(f'Max value          : {np.nanmax(events):.1f} mm')


## 11. Validation — empirical RP from stochastic set vs. IDF

In [ ]:
# ── Load IDF return values for comparison ─────────────────────────────────────
idf_path = os.path.join(IDF_DIR, f'idf_{DURATION_H}h.nc')
has_idf  = os.path.exists(idf_path)

if has_idf:
    ds_idf = xr.open_dataset(idf_path)
    da_idf = ds_idf['return_value'].sel(return_period=TARGET_RP).load()
    ds_idf.close()
    # Clip to sub-domain
    da_idf_d = da_idf.sel(lat=slice(DOMAIN_LAT_MIN, DOMAIN_LAT_MAX),
                           lon=slice(DOMAIN_LON_MIN, DOMAIN_LON_MAX))
    idf_vals = da_idf_d.isel(
        lat=slice(None, None, SUBSAMPLE_N),
        lon=slice(None, None, SUBSAMPLE_N),
    ).values.ravel()
else:
    print('IDF file not found — skipping comparison plot.')
    idf_vals = None

# ── Compute empirical RP from simulated set ────────────────────────────────────
# For each cell: RP_sim(x) = N_sim / rank of x within sorted events
events_sorted = np.sort(events, axis=0)[::-1]   # descending
rp_target_depth_sim = np.nanpercentile(events,
    100 * (1 - 1.0 / TARGET_RP), axis=0)         # RP100 quantile per cell

# ── Scatter plot: IDF RP100 vs simulated RP100 ────────────────────────────────
fig, axes = plt.subplots(1, 2 if has_idf else 1, figsize=(15, 6))
ax_hist = axes[0] if has_idf else axes

ax_hist.hist(events.ravel()[~np.isnan(events.ravel())],
             bins=80, color='steelblue', alpha=0.75, edgecolor='white', lw=0.3)
ax_hist.axvline(np.nanmean(rp_target_depth_sim), color='red', ls='--',
                label=f'Mean RP{TARGET_RP} depth: {np.nanmean(rp_target_depth_sim):.1f} mm')
ax_hist.set_xlabel('Simulated event depth [mm]', fontsize=11)
ax_hist.set_ylabel('Count', fontsize=11)
ax_hist.set_title(f'Distribution of simulated event depths
{N_SIMULATIONS} events × {n_cells} cells', fontsize=11)
ax_hist.legend()
ax_hist.grid(alpha=0.3)

if has_idf and idf_vals is not None:
    idf_flat = idf_vals[~np.isnan(idf_vals)]
    sim_flat = rp_target_depth_sim[~np.isnan(rp_target_depth_sim)][:len(idf_flat)]
    ax_sc = axes[1]
    ax_sc.scatter(idf_flat[:len(sim_flat)], sim_flat, alpha=0.3, s=8, c='steelblue')
    mn = min(idf_flat.min(), sim_flat.min())
    mx = max(idf_flat.max(), sim_flat.max())
    ax_sc.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='1:1 line')
    ax_sc.set_xlabel(f'IDF RP{TARGET_RP} depth [mm]', fontsize=11)
    ax_sc.set_ylabel(f'Simulated RP{TARGET_RP} depth [mm]', fontsize=11)
    ax_sc.set_title(f'Validation: IDF vs. simulated RP{TARGET_RP}', fontsize=11)
    ax_sc.legend()
    ax_sc.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'validation_RP{TARGET_RP}_{DURATION_H}h.png'), dpi=150)
plt.show()
print('Validation plot saved.')


## 12. Map a sample event and the ensemble RP100 field

In [ ]:
# Reshape selected event and ensemble RP map back to (lat_s, lon_s)
LON_M, LAT_M = np.meshgrid(lon_s, lat_s)

# Pick the event whose domain-average is closest to the RP100 level
domain_mean = np.nanmean(events, axis=1)
target_val  = np.nanpercentile(domain_mean, 100 * (1 - 1.0 / TARGET_RP))
idx_sample  = int(np.argmin(np.abs(domain_mean - target_val)))

event_map   = events[idx_sample].reshape(n_lat_s, n_lon_s)
rp_map      = rp_target_depth_sim.reshape(n_lat_s, n_lon_s)

fig, axes = plt.subplots(1, 2, figsize=(16, 7),
    subplot_kw={'projection': ccrs.PlateCarree()} if HAS_CARTOPY else {})

for ax, (data, title) in zip(axes, [
    (event_map, f'Sample stochastic event #{idx_sample}\n(domain-mean ≈ RP{TARGET_RP}) [mm]'),
    (rp_map,    f'Ensemble RP{TARGET_RP} field from {N_SIMULATIONS} simulations [mm]'),
]):
    if HAS_CARTOPY:
        ax.add_feature(cfeature.BORDERS, linewidth=0.5)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.LAND, facecolor='#f0ede6', zorder=0)
    im = ax.pcolormesh(LON_M, LAT_M, data, cmap='Blues', shading='auto')
    plt.colorbar(im, ax=ax, shrink=0.7, label='mm')
    ax.set_title(title, fontsize=11)
    ax.set_extent([DOMAIN_LON_MIN - 0.2, DOMAIN_LON_MAX + 0.2,
                   DOMAIN_LAT_MIN - 0.2, DOMAIN_LAT_MAX + 0.2],
                  crs=ccrs.PlateCarree() if HAS_CARTOPY else None)

plt.suptitle(f'Stochastic Event Set — {DURATION_H}h annual maxima — copula simulation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'stochastic_event_maps_{DURATION_H}h.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Event maps saved.')


## 13. Save event set to NetCDF

In [ ]:
ds_events = xr.Dataset({
    'event_depth': xr.DataArray(
        events.reshape(N_SIMULATIONS, n_lat_s, n_lon_s).astype(np.float32),
        dims=['event', 'lat', 'lon'],
        coords={'event': np.arange(N_SIMULATIONS), 'lat': lat_s, 'lon': lon_s},
        attrs={'units': 'mm',
               'description': f'Stochastic event set — {DURATION_H}h annual max depth'}
    ),
    'rp_map': xr.DataArray(
        rp_map.astype(np.float32),
        dims=['lat', 'lon'],
        coords={'lat': lat_s, 'lon': lon_s},
        attrs={'units': 'mm',
               'description': f'Empirical RP{TARGET_RP} from stochastic simulation'}
    ),
}, attrs={
    'duration_h': DURATION_H,
    'n_simulations': N_SIMULATIONS,
    'variogram_model': VARIOGRAM_MODEL,
    'variogram_range_km': float(range_fit),
    'variogram_sill': float(sill_fit),
    'variogram_nugget': float(nugget_fit),
    'practical_range_km': float(practical_range),
    'validity_limit_km': float(practical_range),
    'domain_lat': f'{DOMAIN_LAT_MIN}–{DOMAIN_LAT_MAX}',
    'domain_lon': f'{DOMAIN_LON_MIN}–{DOMAIN_LON_MAX}',
    'random_seed': RANDOM_SEED,
})

out_nc = os.path.join(OUTPUT_DIR, f'stochastic_events_{DURATION_H}h.nc')
ds_events.to_netcdf(out_nc)
print(f'Saved: {out_nc}')


## 14. Spatial validity assessment

Explicit documentation of where the stochastic simulation is and is not valid.


In [ ]:
print('=' * 65)
print('SPATIAL VALIDITY ASSESSMENT — STOCHASTIC EVENT SET')
print('=' * 65)
print()
print(f'Duration              : {DURATION_H}h')
print(f'Variogram model       : {VARIOGRAM_MODEL}')
print(f'Fitted range          : {range_fit:.1f} km  (correlation scale)')
print(f'Practical range (95%) : {practical_range:.1f} km')
print()
print('VALIDITY LIMITS:')
print(f'  Spatial stationarity: domain must be < {practical_range:.0f} km in extent')
print(f'  Your domain diagonal: {diag_km:.0f} km  —  ',
      end='')
if diag_km <= practical_range:
    print('WITHIN validity range ✓')
else:
    print('EXCEEDS validity range — consider sub-domains!')
print()
print('Physical constraints:')
print('  1. Stationarity assumed: variogram does not vary with location.')
print('     VIOLATED by: mountain-to-plain transitions, coast-inland gradients.')
print('     Mitigation: fit separate variograms per climate sub-zone.')
print()
print('  2. Gaussian copula: tail dependence = 0 (cells become independent')
print('     at extremes).  For heavy-tail joint behaviour use Student-t copula.')
print('     Mitigation: replace L@xi with multivariate-t sampling.')
print()
print('  3. Stationarity in time: GEV parameters assumed stationary 1991–2020.')
print('     Non-stationarity due to climate change not captured.')
print()
print('RECOMMENDED APPLICATION:')
print(f'  – Single climate zone, diameter ≤ {min(practical_range, 150):.0f} km')
print('  – Urban drainage / catchment design storm frequency analysis')
print('  – NOT for simultaneous national-scale risk assessment')
